In [ ]:
from IPython.display import clear_output



clear_output()
import kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

%matplotlib inline

In [ ]:



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# check missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df = df.fillna(df.mean())

In [ ]:
# Task 2: Write your code here:
# check duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# check categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 5: Write your code here:
#  check If the target imbalanced?
def check_target_imbalance(df):
  print("Target Distribution:")

  df["Target"].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df)

In [ ]:
# Import model
%pip install  catboost xgboost  imbalanced-learn -q



In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Task 1: Write your code here:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X=df.drop("Target", axis=1)
y=df['Target']
model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )

In [ ]:
catboost = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models


  print("Training catboost ...")

    # Fit the model on train data
  model.fit(X_train, y_train)

    # Use the model to predict the test data
  y_pred = model.predict(X_test)

    # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  catboost['accuracy'].append(accuracy)
  catboost['f1'].append(f1)

In [ ]:
# Task 2,3,4,5: Write your code here:

print(f"  Accuracy:  {np.mean(catboost['accuracy']):.4f}")
print(f"  F1-Score:  {np.mean(catboost ['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(30, 12))
plt.barh(feature_importance['feature'][0:10], feature_importance['importance'][0:10])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

# name of Top 10 Feature importance
print('name of Top 10 Feature importance\n')
for indx ,i in enumerate(feature_importance['importance'][0:10]):

  print(f' {feature_importance['feature'][indx]} has W : {i}')




In [ ]:
# Task Bonus: Write your code here:
golden = ['P_2','D_39','B_1','B_2','R_1','S_3','D_41','B_3','D_42','D_43']
X_new=df[golden]
X_new

In [ ]:
catboost = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
for fold_idx, (train_index, test_index) in enumerate(skf.split(X_new, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # Get the train & test split for this fold
  X_train, X_test = X_new.iloc[train_index], X_new.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models


  print("Training catboost ...")

    # Fit the model on train data
  model.fit(X_train, y_train)

    # Use the model to predict the test data
  y_pred = model.predict(X_test)

    # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  catboost['accuracy'].append(accuracy)
  catboost['f1'].append(f1)

In [ ]:
print(f"  Accuracy:  {np.mean(catboost['accuracy']):.4f}")
print(f"  F1-Score:  {np.mean(catboost ['f1']):.4f}")